In [1]:
import pathlib
import os
import pandas as pd
import numpy as np

In [2]:
!pip install biopython


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 9.0 MB/s eta 0:00:00


In [3]:
# Set up file system
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [4]:
# download the git repo
!git clone https://github.com/ConorMesser/stats_305c.git
!cd stats_305c/; git pull

Cloning into 'stats_305c'...
remote: Enumerating objects: 59, done.
remote: Counting objects: 100% (59/59), done.
remote: Compressing objects: 100% (42/42), done.
remote: Total 59 (delta 24), reused 45 (delta 15), pack-reused 0 (from 0)
Receiving objects: 100% (59/59), 14.05 MiB | 8.81 MiB/s, done.
Resolving deltas: 100% (24/24), done.
Already up to date.


# Clean MIC Data (MOVE BACK TO OTHER NOTEBOOK)

In [5]:
file_path = '/content/drive/MyDrive/Courses/STATS305c/data' # Conor

activities_original = pd.read_csv(os.path.join(file_path,'MIC_data_original.csv'))

In [92]:
track_row_numbers = {}
cleaned_df = activities_original.copy()
track_row_numbers['original'] = cleaned_df.shape[0]
# remove missing from 188575 to 177258
cleaned_df = cleaned_df.dropna(subset=["Peptide ID",
                                       "Target Species",
                                       "Activity Measure",
                                       "Activity",
                                       "Unit"])
track_row_numbers['remove_NAs'] = cleaned_df.shape[0]

#remove other measurement values such as MBC, IC50, LC, MIC50  from 177258 to 135441
cleaned_df = cleaned_df[cleaned_df["Activity Measure"] == "MIC"]
track_row_numbers['only_MIC'] = cleaned_df.shape[0]

# length filter from 97970 to 89765
cleaned_df["peptide_length"] = cleaned_df["Peptide Sequence"].str.len()
cleaned_df = cleaned_df[(cleaned_df["peptide_length"] >= 8)]
track_row_numbers['length_ge_8'] = cleaned_df.shape[0]
# Don't filter out long peptides for now
# cleaned_df = cleaned_df[(cleaned_df["peptide_length"] <= 50)]
# track_row_numbers['length_le_50'] = cleaned_df.shape[0]

# Remove peptides with unknown amino acids (X/x)
# I think not - these should be able to handled by ESM and our features still
# cleaned_df = cleaned_df[cleaned_df["Peptide Sequence"].str.match("^[ACDEFGHIKLMNPQRSTVWY]+$", na=False)]
# track_row_numbers['known_amino_acids'] = cleaned_df.shape[0]

# drop duplicates from 63532 to 61715
# Do we think these duplicates are accidental? Or represent true repeated tests?
# cleaned_df = cleaned_df.drop_duplicates()
# track_row_numbers['remove_duplicates'] = cleaned_df.shape[0]


In [93]:

track_row_numbers

{'original': 188575,
 'remove_NAs': 177258,
 'only_MIC': 135441,
 'length_ge_8': 119205}

In [94]:
# 1. Strip tabs and remove '='
cleaned_df.loc[:, 'Activity'] = cleaned_df['Activity'].str.replace(r'\s+', '', regex=True)
cleaned_series = cleaned_df['Activity'].str.strip('\t').str.replace('=', '', regex=False)

# 2. Convert ≥ to > and strip surrounding whitespace
cleaned_series = cleaned_series.str.replace(r'\s*≥\s*', '>', regex=True)

# 3. Convert ≤ to < and strip surrounding whitespace
cleaned_series = cleaned_series.str.replace(r'\s*≤\s*', '<', regex=True)

# 4. Strip whitespace around remaining operational characters (>, <)
cleaned_series = cleaned_series.str.replace(r'\s*([><])\s*', r'\1', regex=True)

# 5. Clean any trailing or leading general whitespace left over
cleaned_df.loc[:, 'Activity'] = cleaned_series.str.strip().values

cleaned_df.loc[:, 'Activity_raw'] = cleaned_df.loc[:, 'Activity']

dash_pattern = r'[±+]'
plus_minus = cleaned_df['Activity'].str.contains(dash_pattern)
cleaned_df['plus_minus'] = plus_minus
cleaned_df['Activity_std'] = 0
cleaned_df.loc[plus_minus, ['Activity', 'Activity_std']] = cleaned_df.loc[plus_minus, 'Activity'].str.split(dash_pattern, expand=True).values

cleaned_df['Activity_low'] = cleaned_df['Activity']
cleaned_df['Activity_low'] = cleaned_df['Activity_low'].str.replace('–', '-')
ranges = cleaned_df['Activity_low'].str.contains('-')
cleaned_df['range'] = ranges
cleaned_df.loc[ranges, ['Activity_low', 'Activity']] = cleaned_df.loc[ranges, 'Activity_low'].str.split('-', expand=True).values

# clean up values where Activity == 0
zero_mask = cleaned_df['Activity_low'] == '0'
cleaned_df.loc[zero_mask, 'Activity_low'] = cleaned_df.loc[zero_mask, 'Activity']
cleaned_df.loc[zero_mask, 'range'] = False

/tmp/ipykernel_6474/983935643.py:23: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['5.6', '2.5', '0.5', '0.2', '3', '2', '2', '2', '2', '1', '3', '0.4', '1.5', '0.5', '0.1', '4', '9.50', '12.75', '9.50', '45.77', '0.3', '0.4', '1.0', '0.2', '0.1', '0.6', '3', '37', '3', '4', '3', '3', '4', '3', '7', '1.2', '1.4', '1.4', '2.4', '12.5', '0.8', '7.9', '10.50', '5.22', '0.12', '8.52', '16.22', '3.12', '0.25', '6.34', '1.0', '2.6', '0.014', '0.014', '0.028', '0.028', '0.014', '0.014', '0.038', '0.028', '0.014', '1.2', '19.5', '1.0', '12.4', '9.2', '0.6', '0.008', '0.005', '0.013', '0.025', '0.024', '0.075', '0.014', '0.079', '0.136', '0.018', '0.129', '0.006', '0.027', '0.015', '0.009', '0.010', '0.011', '0.015', '0.038', '0.012', '0.003', '0.10', '0.03', '0.21', '0.14', '0.60', '16.6', '7', '1.5', '1.6', '0.8', '24.8', '6.2', '3.1', '4.9', '9.8', '0.9', '0.7', '0.6', '0.6', '0.7', '0.8', '0.9', '0.7', '0.6'

In [95]:

# Clean up ranges that have a greater than in the low value
# This is already the definition of a range
activity_gt = cleaned_df['Activity_low'].str.contains('>')
cleaned_df.loc[activity_gt & ranges, 'Activity_low'] = cleaned_df.loc[activity_gt & ranges, 'Activity_low'].str.strip('>')

# Assume values with less than in a range are a mistake
activity_lt = cleaned_df['Activity_low'].str.contains('<')
cleaned_df.loc[activity_lt & ranges, 'Activity_low'] = cleaned_df.loc[activity_lt & ranges, 'Activity_low'].str.strip('<')

# drop weird value where high activity in range has a less than
thresholded_high = cleaned_df['Activity'].str.contains('<')
cleaned_df = cleaned_df[~(thresholded_high & ranges)]

# What to do about greater than values in high value of range?
# Will assume high value is not censored but that '->' is just an arrow
censored_high = cleaned_df['Activity'].str.contains('>')
cleaned_df.loc[ranges & censored_high, 'Activity'] = cleaned_df.loc[ranges & censored_high, 'Activity'].str.strip(' ').str.strip('>')

# flip order of any ranges with backwards low and high
only_ranges = cleaned_df.loc[ranges].copy()
backwards_values = only_ranges[(only_ranges['Activity'].astype(float) < only_ranges['Activity_low'].astype(float))].index
tmp_activity = only_ranges.loc[backwards_values, 'Activity_low']
cleaned_df.loc[backwards_values, 'Activity_low'] = cleaned_df.loc[backwards_values, 'Activity']
cleaned_df.loc[backwards_values, 'Activity'] = tmp_activity



In [96]:
thresholded_values = cleaned_df['Activity'].str.contains('<')
censored_values = cleaned_df['Activity'].str.contains('>')

# take out < - this is just the definition
remove_lt = cleaned_df.loc[~ranges & thresholded_values, 'Activity'].str.strip('<').values
cleaned_df.loc[~ranges & thresholded_values, 'Activity_low'] = remove_lt
cleaned_df.loc[~ranges & thresholded_values, 'Activity'] = remove_lt

# add columns for censored values (and then take out the >)
cleaned_df['censored_high'] = False
cleaned_df.loc[~ranges & censored_values, 'censored_high'] = True
cleaned_df.loc[~ranges & censored_values, 'Activity_low'] = cleaned_df.loc[~ranges & censored_values, 'Activity_low'].str.strip('>').values
cleaned_df.loc[~ranges & censored_values, 'Activity'] = cleaned_df.loc[~ranges & censored_values, 'Activity'].str.strip('>').values

# apply /2 to all activity values that don't have a range (this assumes testing by doubling)
cleaned_df = cleaned_df[~cleaned_df.loc[:, 'Activity_low'].str.contains(',')]
cleaned_df.loc[~ranges & ~censored_values, 'Activity_low'] = cleaned_df.loc[~ranges & ~censored_values, 'Activity_low'].astype(float) / 2

cleaned_df['Activity_low'] = cleaned_df['Activity_low'].astype(float)
cleaned_df['Activity'] = cleaned_df['Activity'].astype(float)


In [97]:
(cleaned_df.loc[ranges, 'Activity'] / cleaned_df.loc[ranges, 'Activity_low']).describe()

,0
count,6420.000000
mean,8.005873
std,294.074100
min,1.000000
25%,2.000000
50%,2.000000
75%,2.000000
max,23333.333333


In [99]:

(cleaned_df.loc[ranges, 'Activity'] / cleaned_df.loc[ranges, 'Activity_low']).value_counts().iloc[:10]

,count
2.000000,4463
4.000000,307
2.016129,285
8.000000,156
1.500000,84
1.937500,81
2.500000,78
16.000000,49
2.083333,45
1.250000,44


In [100]:

cleaned_df.columns

Index(['Peptide ID', 'Peptide Sequence', 'Target Species', 'Activity Measure',
       'Activity', 'Unit', 'pH', 'Ionic Strength mM', ' "Salt Type"', 'Medium',
       'CFU', 'Note', 'Reference', 'peptide_length', 'Activity_raw',
       'plus_minus', 'Activity_std', 'Activity_low', 'range', 'censored_high'],
      dtype='object')

In [102]:
print(f"Number of Range values (raw had -): {cleaned_df['range'].sum()}")
print(f"Number of Censored values (>): {cleaned_df['censored_high'].sum()}")

top5_counts = cleaned_df.groupby('Unit')['Activity'].value_counts().groupby(level=0).head(5)
total_counts = cleaned_df.groupby('Unit')['Activity'].count()

top5_sum = top5_counts.groupby(level=0).sum()
ratio = top5_sum / total_counts

print("Top 5 activity values as fraction of total, by unit:")
print(ratio)

censored_counts = cleaned_df[cleaned_df['censored_high']]['Activity'].value_counts()
print(censored_counts.head(5))

Number of Range values (raw had -): 6418
Number of Censored values (>): 23839
Top 5 activity values as fraction of total, by unit:
Unit
µM       0.287461
µg/ml    0.364871
dtype: float64
Activity
128.0    4255
100.0    3593
64.0     3377
50.0     1887
32.0     1598
Name: count, dtype: int64


In [106]:
((df_conv["Peptide Sequence"].str.count('x') + df_conv["Peptide Sequence"].str.count('X')) > 0).sum()

np.int64(18945)

In [116]:
((df_conv["Peptide Sequence"].str.count('\*')) > 0).sum()

<>:1: SyntaxWarning: invalid escape sequence '\*'
<>:1: SyntaxWarning: invalid escape sequence '\*'
/tmp/ipykernel_6474/3247263387.py:1: SyntaxWarning: invalid escape sequence '\*'
  ((df_conv["Peptide Sequence"].str.count('\*')) > 0).sum()


np.int64(0)

In [120]:

from Bio.SeqUtils import molecular_weight
from Bio.Seq import Seq
from Bio.Data.IUPACData import protein_letters


def peptide_mw(seq):
    try:
        return molecular_weight(Seq(seq), seq_type='protein')
    except Exception:
        return None

df_conv = cleaned_df.copy()

# only work on rows with numeric Activity - maybe add mean molecular weight times number of Xs
df_conv['num_X_O'] = df_conv["Peptide Sequence"].str.count('x') + df_conv["Peptide Sequence"].str.count('X')
df_conv["MW"] = df_conv["Peptide Sequence"].str.replace('X','').str.replace('x','').apply(peptide_mw)

mean_aa_weight = peptide_mw(protein_letters)
df_conv["MW"] = df_conv["MW"] + mean_aa_weight * df_conv['num_X_O']



In [124]:

# check rows where MW couldn't be calculated
print('Unknown MW:', df_conv[df_conv["MW"].isna() | (df_conv["MW"] <= 0)].shape[0])

# convert µg/ml → µM; keep µM rows as-is
df_conv["Activity_uM"] = df_conv.apply(
    lambda row: (row["Activity"] / row["MW"]) * 1000
    if row["Unit"].strip() == "µg/ml"
    else row["Activity"],
    axis=1
)

# convert µg/ml → µM; keep µM rows as-is
df_conv["Activity_low_uM"] = df_conv.apply(
    lambda row: (row["Activity_low"] / row["MW"]) * 1000
    if row["Unit"].strip() == "µg/ml"
    else row["Activity_low"],
    axis=1
)

print(df_conv["Unit"].value_counts())
print(df_conv[["Peptide Sequence", "Activity", "Unit", "MW", "Activity_uM"]].head(10))

print(df_conv.shape[0])

Unknown MW: 0
Unit
µM       64774
µg/ml    54422
Name: count, dtype: int64
            Peptide Sequence  Activity Unit         MW  Activity_uM
0  FLGLIFHGLVHAGKLIHGLIHRNRG       1.5   µM  2776.2926          1.5
1  FLGLIFHGLVHAGKLIHGLIHRNRG       7.5   µM  2776.2926          7.5
2  FLGLIFHGLVHAGKLIHGLIHRNRG       7.5   µM  2776.2926          7.5
3  FLGLIFHGLVHAGKLIHGLIHRNRG      15.0   µM  2776.2926         15.0
4  FLGLIFHGLVHAGKLIHGLIHRNRG      48.0   µM  2776.2926         48.0
5  FLGLIFHGLVHAGKLIHGLIHRNRG       3.0   µM  2776.2926          3.0
6  FLGLIFHGLVHAGKLIHGLIHRNRG       6.0   µM  2776.2926          6.0
7  FLGLIFHGLVHAGKLIHGLIHRNRG       7.5   µM  2776.2926          7.5
8  FLGLIFHGLVHAGKLIHGLIHRNRG      12.0   µM  2776.2926         12.0
9  FLGLIFHGLVHAGKLIHGLIHRNRG      15.0   µM  2776.2926         15.0
119196


In [127]:
df_conv['Peptide ID'].nunique()

16968

In [129]:
df_conv['Target Species'].nunique()

5586

In [130]:

df_conv.to_csv('/content/drive/MyDrive/Courses/STATS305c/data/MIC_data_cleaned.csv', index=False)

# Load Data

In [328]:
# file_path = '/content/drive/MyDrive/STATS305c/data' # Lauren
file_path = '/content/drive/MyDrive/Courses/STATS305c/data' # Conor

mic_file = pathlib.Path(os.path.join(file_path, 'MIC_data_cleaned.csv'))

full_mic_data = pd.read_csv(mic_file)


# Functions

In [330]:
taxonomy_input_df = full_mic_data[['Target Species']].copy()
taxonomy_input_df.rename(columns={'Target Species': 'strain'}, inplace=True)
taxonomy_input_df.loc[:, 'species_input'] = taxonomy_input_df.loc[:, 'strain'].str.split().str[:2].str.join(' ')

In [331]:
from Bio import Entrez
# NCBI requires an email address to use their public API
Entrez.email = "csmesser@stanford.edu"
Entrez.api_key = ""  # FILL IN

import time

def get_taxonomy_batched(species_list):
    tax_ids = []
    tax_ids_dict = {}
    failed_species = []

    print("Step 1: Fetching TaxIDs...")
    for species in species_list:
        try:
            handle = Entrez.esearch(db="taxonomy", term=species)
            record = Entrez.read(handle)
            handle.close()

            if record["IdList"]:
                tax_ids.append(record["IdList"][0])
                tax_ids_dict[record["IdList"][0]] = species
            else:
                failed_species.append(species)
        except Exception as e:
            failed_species.append(species)

        time.sleep(0.11) # Respect NCBI API limits

    # SAFETY CHECK: If no IDs were found, exit early to prevent a 400 error
    if not tax_ids:
        print("No valid TaxIDs found. Exiting.")
        return pd.DataFrame(), failed_species

    print(f"Step 2: Fetching lineages for {len(tax_ids)} species in chunks...")

    records = []
    chunk_size = 100 # NCBI handles chunks of 100 perfectly

    print(tax_ids)

    # CHUNKING LOGIC: Process the IDs in batches of 100
    for i in range(0, len(tax_ids), chunk_size):
        chunk = tax_ids[i:i + chunk_size]
        id_string = ",".join(chunk)

        try:
            fetch_handle = Entrez.efetch(db="taxonomy", id=id_string, retmode="xml")
            chunk_records = Entrez.read(fetch_handle)
            fetch_handle.close()

            # Entrez.read returns a list of dictionaries; add them to our master list
            records.extend(chunk_records)

        except Exception as e:
            print(f"Failed to fetch chunk {i} to {i+chunk_size}: {e}")

        time.sleep(0.11) # Sleep between chunks

    print("Step 3: Parsing data...")
    target_ranks = ['superkingdom', 'domain', 'kingdom', 'phylum', 'class', 'order', 'family', 'genus', 'species']
    results = []

    for tax_data in records:
        row_data = {rank: pd.NA for rank in target_ranks}
        current_name = tax_data.get("ScientificName", "")
        this_tax_id = tax_data.get("TaxId", "")
        row_data['input'] = tax_ids_dict[this_tax_id]

        ancestors = tax_data.get("LineageEx", [])
        all_nodes = ancestors + [{"Rank": tax_data.get("Rank", ""), "ScientificName": current_name}]

        for node in all_nodes:
            rank = node.get("Rank", "").lower()
            name = node.get("ScientificName", "")
            if rank in target_ranks:
                row_data[rank] = name

        results.append(row_data)

    df = pd.DataFrame(results)

    return df, failed_species


In [332]:

# ==========================================
# Example Usage
# ==========================================
# A mix of bacteria, fungi, viruses, and generic IDs
print("Querying NCBI...\n")

# Generate the dataframe
taxonomy_df, failed_species = get_taxonomy_batched(taxonomy_input_df['species_input'].unique())


Querying NCBI...

Step 1: Fetching TaxIDs...
Step 2: Fetching lineages for 732 species in chunks...
['1280', '1283', '29385', '1292', '1423', '562', '28901', '582', '1352', '5476', '4932', '573', '550', '287', '584', '1282', '1351', '36911', '109871', '714', '470', '40324', '546', '5480', '5482', '4909', '1309', '1579', '53413', '585', '5478', '1824', '1747', '571', '646', '1311', '46126', '1933880', '1334', '1349', '146827', '1336', '1396', '1288', '644', '294', '1377', '1270', '669', '55601', '303', '1799160', '1428', '1639', '90371', '1773', '1502', '552', '317', '339', '48664', '56448', '210', '520', '5580', '5059', '746128', '5061', '5507', '5530', '40559', '36656', '100870', '5465', '76777', '727', '615', '5207', '5141', '5516', '984957', '1404', '40215', '633', '29388', '1314', '4896', '55194', '5553', '1305', '1310', '1302', '1656', '1655', '1582', '1613', '29908', '47879', '5599', '169388', '29918', '28447', '554', '56460', '216816', '1304', '1307', '1313', '137621', '666', '6

In [333]:
taxonomy_df.head()

,superkingdom,domain,kingdom,phylum,class,order,family,genus,species,input
0,<NA>,Bacteria,Bacillati,Bacillota,Bacilli,Caryophanales,Staphylococcaceae,Staphylococcus,Staphylococcus aureus,Staphylococcus aureus
1,<NA>,Bacteria,Bacillati,Bacillota,Bacilli,Caryophanales,Staphylococcaceae,Staphylococcus,Staphylococcus haemolyticus,Staphylococcus haemolyticus
2,<NA>,Bacteria,Bacillati,Bacillota,Bacilli,Caryophanales,Staphylococcaceae,Staphylococcus,Staphylococcus saprophyticus,Staphylococcus saprophyticus
3,<NA>,Bacteria,Bacillati,Bacillota,Bacilli,Caryophanales,Staphylococcaceae,Staphylococcus,Staphylococcus warneri,Staphylococcus warneri
4,<NA>,Bacteria,Bacillati,Bacillota,Bacilli,Caryophanales,Bacillaceae,Bacillus,Bacillus subtilis,Bacillus subtilis


In [334]:
taxonomy_df.shape

(732, 10)

In [335]:
len(failed_species)

48

In [336]:
sorted(failed_species)

['Agrobacterium rhizogenes',
 'Bacillus circulans',
 'Bacteroides vulgatus',
 'Candida guilliermondii',
 'Clavibacter fangii',
 'Clostridium oroticum',
 'Cryptococcus albidus',
 'Cryptococcus cuniculi',
 'Enterobacter spp.',
 'Eubacterium rectale',
 'Gibberella saubinetii',
 'Haemophilus spp.',
 'Human acute',
 'Human breast',
 'Human cervical',
 'Human gastric',
 'Human lung',
 'Human myelogenous',
 'Human ovarian',
 'Human pancreatic',
 'Human prostate',
 'Human skin',
 'Lactococcus raffinolactis',
 'Moraxella spp.',
 'Mycobacterium smegmatis',
 'Mycoplasma hominis',
 'Neisseria spp.',
 'Nocardia spp.',
 'Paecilomyces spp.',
 'Parageobacillus toebi',
 'Peptostreptococcus micros',
 'Prevotella copri',
 'Pseudomonas oleovorans',
 'Pseudomonas stutzeri',
 'Rhodococcus equi',
 'Rhodococcus fascians',
 'Rhodococcus sp.',
 'Salmonella Bazenheid',
 'Salmonella bonariensis',
 'Serratia sp.',
 'Slime mold',
 'Staphylococcus citreus',
 'Staphylococcus sciuri',
 'Streptococcus Sc181',
 'Strepto

In [337]:
taxonomy_output_df = taxonomy_input_df.merge(taxonomy_df, how='left', left_on='species_input', right_on='input')


In [338]:
sorted(taxonomy_output_df[taxonomy_output_df['genus'].isnull()]['species_input'].unique())

['Acinetobacter junii',
 'Aeromonas caviae',
 'Agrobacterium rhizogenes',
 'Bacillus circulans',
 'Bacteroides vulgatus',
 'Candida guilliermondii',
 'Candida kefyr',
 'Candida sp.',
 'Clavibacter fangii',
 'Clostridium oroticum',
 'Cryptococcus albidus',
 'Cryptococcus cuniculi',
 'Enterobacter spp.',
 'Eubacterium rectale',
 'Gibberella saubinetii',
 'Haemophilus influenzae',
 'Haemophilus spp.',
 'Human acute',
 'Human breast',
 'Human cervical',
 'Human gastric',
 'Human lung',
 'Human myelogenous',
 'Human ovarian',
 'Human pancreatic',
 'Human prostate',
 'Human skin',
 'Hypocreales sp.',
 'Lactococcus raffinolactis',
 'Moraxella spp.',
 'Mycobacterium smegmatis',
 'Mycoplasma hominis',
 'Neisseria spp.',
 'Nocardia spp.',
 'Paecilomyces spp.',
 'Parageobacillus toebi',
 'Peptostreptococcus micros',
 'Phytophthora nicotianae',
 'Prevotella copri',
 'Pseudomonas oleovorans',
 'Pseudomonas stutzeri',
 'Rhodococcus equi',
 'Rhodococcus fascians',
 'Rhodococcus sp.',
 'Salmonella Baz

In [339]:
{val: '' for val in taxonomy_output_df[taxonomy_output_df['genus'].isna()]['species_input'].value_counts().sort_values().iloc[-18:][::-1].index}

{'Mycobacterium smegmatis': '',
 'Bacteroides vulgatus': '',
 'Candida kefyr': '',
 'Streptococcus group': '',
 'Haemophilus influenzae': '',
 'Eubacterium rectale': '',
 'Prevotella copri': '',
 'Staphylococcus sciuri': '',
 'Streptococcus Sc181': '',
 'Pseudomonas stutzeri': '',
 'Slime mold': '',
 'Candida guilliermondii': '',
 'Acinetobacter junii': '',
 'Serratia sp.': '',
 'Rhodococcus equi': '',
 'Aeromonas caviae': '',
 'Peptostreptococcus micros': '',
 'hotobacterium damselae': ''}

In [340]:
na_species = {  # From Gemini
    # --- Recently reclassified ---
    'Mycobacterium smegmatis': 'Mycolicibacterium smegmatis',
    'Bacteroides vulgatus': 'Phocaeicola vulgatus',
    'Candida kefyr': 'Kluyveromyces marxianus',
    'Eubacterium rectale': 'Agathobacter rectalis',
    'Prevotella copri': 'Segatella copri',
    'Staphylococcus sciuri': 'Mammaliicoccus sciuri',
    'Pseudomonas stutzeri': 'Stutzerimonas stutzeri',
    'Candida guilliermondii': 'Meyerozyma guilliermondii',
    'Rhodococcus equi': 'Prescottia equi',
    'Peptostreptococcus micros': 'Parvimonas micra',

    # --- Formatting, Typos, and Unresolved terms ---
    'Streptococcus group': 'Streptococcus',
    'Streptococcus Sc181': 'Streptococcus',
    'Slime mold': 'Mycetozoa',
    'Serratia sp.': 'Serratia',
    'hotobacterium damselae': 'Photobacterium damselae',

    # --- Capitalization fixed (Already valid in NCBI) ---
    'Haemophilus influenzae': 'Haemophilus influenzae',
    'Acinetobacter junii': 'Acinetobacter junii',
    'Aeromonas caviae': 'Aeromonas caviae'

}

taxonomy_failed_df, failed_x2_species = get_taxonomy_batched(na_species.values())


Step 1: Fetching TaxIDs...
Step 2: Fetching lineages for 17 species in chunks...
['1772', '821', '4911', '39491', '165179', '1296', '316', '4929', '33033', '1301', '1301', '142796', '2985502', '38293', '727', '40215', '648']
Step 3: Parsing data...


In [341]:
tmp = taxonomy_failed_df.copy()
tmp['input'] = tmp['input'].map({v: k for k, v in na_species.items()})
tax_df_w_failed = pd.concat([taxonomy_df, tmp],axis=0).drop_duplicates()

# Add Streptococcus group values
tax_df_w_failed.loc[len(tax_df_w_failed)] = [np.nan, 'Bacteria', 'Bacillati', 'Bacillota', 'Bacilli',
       'Lactobacillales', 'Streptococcaceae', 'Streptococcus',
       np.nan, 'Streptococcus group']

taxonomy_output_df = taxonomy_input_df.merge(tax_df_w_failed, how='left', left_on='species_input', right_on='input')
taxonomy_output_df = taxonomy_output_df[['superkingdom','domain','kingdom','phylum','class','order','family','genus','species','strain']].bfill(axis=1)


In [346]:
taxonomy_output_df.to_csv(os.path.join(file_path, 'species_taxonomy.tsv'), sep='\t', index=False)